In [ ]:
import anthropic
import json
import os
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def get_completion(model, prompt: str, system_prompt="", prefill="", temp=1.0, max_tokens=10000):
    message = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        temperature=temp,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

In [ ]:
######################################## INPUT VARIABLES ########################################

MODEL_NAME = "claude-sonnet-4-20250514"
NUM_TASKS = 5
DEVICE = "mobile (e.g., smartphones)"
NUM_SOURCES = 3
FREQUENCY = "only once"
INPUT_TYPES = "web pages"
INPUT_SIZE = "large"
OUTPUT_TYPES = "text"
OUTPUT_SIZE = "large (>1024 tokens)"
LATENCY = "none"
USER_PREFERENCES = "minimize data usage"

######################################## PROMPT ELEMENTS ########################################

##### Prompt element 1: `user` role
# Make sure that your Messages API call always starts with a `user` role in the messages array.
# The get_completion() function as defined above will automatically do this for you.

##### Prompt element 2: Task context
# Give Claude context about the role it should take on or what goals and overarching tasks you want it to undertake with the prompt.
# It's best to put context early in the body of the prompt.
TASK_CONTEXT = f"You are tasked with designing a benchmark consisting of {NUM_TASKS} different tasks for evaluating AI web agents. The goal is to measure system resource usage and impacts when an AI agent integrated into a web browser performs the task. This benchmark will help assess the efficiency and performance of AI agents in real-world scenarios."

##### Prompt element 3: Tone context
# If important to the interaction, tell Claude what tone it should use.
# This element may not be necessary depending on the task.
TONE_CONTEXT = ""

##### Prompt element 4: Detailed task description and rules
# Expand on the specific tasks you want Claude to do, as well as any rules that Claude might have to follow.
# This is also where you can give Claude an "out" if it doesn't have an answer or doesn't know.
# It's ideal to show this description and rules to a friend to make sure it is laid out logically and that any ambiguous words are clearly defined.
TASK_DESCRIPTION = f"""Here are the specifications for the task you need to design:
<specifications>
    <target_device>{DEVICE}</target_device>
    <num_sources>{NUM_SOURCES}</num_sources>
    <frequency>{FREQUENCY}</frequency>
    <input_types>{INPUT_TYPES}</input_types>
    <input_size>{INPUT_SIZE}</input_size>
    <output_types>{OUTPUT_TYPES}<output_types>
    <output_size>{OUTPUT_SIZE}</output_size>
    <latency_requirements>{LATENCY}</latency_requirements>
    <user_preferences>{USER_PREFERENCES}</user_preferences>
</specifications>

When designing a benchmark task, follow these guidelines:
1. The task should have an interesting goal that can yield value to humans, whether educational, financial, or even creative/recreational.
2. It should require dedicated manual effort from a human to perform.
3. The task should be resource-intensive for the system.
4. The task should not be a generic task, and should include specific details.
5. Describe the task succinctly but informatively, as if it were for a human.
6. Ensure the task aligns with the given specifications, especially the target device.
7. Consider how the agent will utilize system resources to perform the task."""

##### Prompt element 5: Examples
# Provide Claude with at least one example of an ideal response that it can emulate. Encase this in <example></example> XML tags. Feel free to provide multiple examples.
# If you do provide multiple examples, give Claude context about what it is an example of, and enclose each example in its own set of XML tags.
# Examples are probably the single most effective tool in knowledge work for getting Claude to behave as desired.
# Make sure to give Claude examples of common edge cases. If your prompt uses a scratchpad, it's effective to give examples of how the scratchpad should look.
# Generally more examples = better.
EXAMPLES = """Here are some examples of benchmark tasks in JSON format (not necessarily following the above specifications):
<example>
[
    {
        "user_profile": "A high school student looking for a good college to study Computer Science",
        "task_type": "Web scraping",
        "task_description": "Create a table of all Computer Science professors at the top 10 liberal arts colleges in the US. The tables should be in CSV format with the following columns: Full name, Job title, College, Fields, Personal website."
    },
    {
        "user_profile": "A new homeowner looking for the best bed mattress",
        "task_type": "Product recommendation",
        "task_description": "Find all bed mattresses available for sale online within 500$ and create a detailed report comparing them. The report should include price, material, user reviews, etc."
    },
    {
        "user_profile": "A music enthusiast who wants to share new music on social media",
        "task_type": "Music recommendation and social media management",
        "task_description": "Everyday, search for a new rock or pop song released by any artist, write a short Twitter post describing the song, and post it to my Twitter account."
    },
</example>"""

##### Prompt element 6: Input data to process
# If there is data that Claude needs to process within the prompt, include it here within relevant XML tags.
# Feel free to include multiple pieces of data, but be sure to enclose each in its own set of XML tags.
# This element may not be necessary depending on task. Ordering is also flexible.
INPUT_DATA = ""

##### Prompt element 7: Immediate task description or request #####
# "Remind" Claude or tell Claude exactly what it's expected to immediately do to fulfill the prompt's task.
# This is also where you would put in additional variables like the user's question.
# It generally doesn't hurt to reiterate to Claude its immediate task. It's best to do this toward the end of a long prompt.
# This will yield better results than putting this at the beginning.
# It is also generally good practice to put the user's query close to the bottom of the prompt.
IMMEDIATE_TASK = ""

##### Prompt element 8: Precognition (thinking step by step)
# For tasks with multiple steps, it's good to tell Claude to think step by step before giving an answer
# Sometimes, you might have to even say "Before you give your answer..." just to make sure Claude does this first.
# Not necessary with all prompts, though if included, it's best to do this toward the end of a long prompt and right after the final immediate task request or description.
PRECOGNITION = "Before responding, carefully think about and plan your tasks design. Make sure to consider how the agent would utilize the system's resources to perform the task. Ensure the tasks meets all the guidelines and specifications provided."

##### Prompt element 9: Output formatting
# If there is a specific way you want Claude's response formatted, clearly tell Claude what that format is.
# This element may not be necessary depending on the task.
# If you include it, putting it toward the end of the prompt is better than at the beginning.
OUTPUT_FORMATTING = """Output your response as a JSON array like this:
[
    {
        "user_profile": "User profile",
        "task_type": "Task type",
        "task_description": "Detailed task description"
    },
    {
        "user_profile": "User profile",
        "task_type": "Task type",
        "task_description": "Detailed task description"
    }
]"""

##### Prompt element 10: Prefilling Claude's response (if any)
# A space to start off Claude's answer with some prefilled words to steer Claude's behavior or response.
# If you want to prefill Claude's response, you must put this in the `assistant` role in the API call.
# This element may not be necessary depending on the task.
PREFILL = ""



######################################## COMBINE ELEMENTS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Print full prompt
print(PROMPT)

In [ ]:
raw_result = get_completion(MODEL_NAME, PROMPT)

In [ ]:
start_match = "```json\n"
end_match = "\n```"
parsed_result = json.loads(raw_result[raw_result.find(start_match) + len(start_match):raw_result.rfind(end_match)])
for res in parsed_result:
    print(res)